In [9]:
from pyspark.sql.functions import round, col, dayofmonth, month, year, to_date, quarter, substring,  when, regexp_replace

StatementMeta(, 40cc943d-a31c-4fa1-b881-8fc335be05b3, 11, Finished, Available, Finished, False)

In [10]:
bronze_table_path = "abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Bronze.Lakehouse/Tables/dbo/wind_power"

df = spark.read.format('delta').load(bronze_table_path)



StatementMeta(, 40cc943d-a31c-4fa1-b881-8fc335be05b3, 12, Finished, Available, Finished, False)

In [11]:

df_transformed = (
    df.withColumn("energy_produced", round(col("energy_produced"),2))
    .withColumn("wind_speed", round(col("wind_speed"),2))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("year", year(col("date")))
    .withColumn("time", regexp_replace(col("time"),"-",":") )
    .withColumn("hour_of_day",substring(col("time"),1,2  ).cast("int"))
    .withColumn("mint_of_hour",substring(col("time"),4,2  ).cast("int"))
    .withColumn("sec_of_minute",substring(col("time"),7,2  ).cast("int"))
    .withColumn("time_period", when((col("hour_of_day") >= 5 ) & (col("hour_of_day") < 12 ), "Morning" )
                               .when((col("hour_of_day") >= 12 ) & (col("hour_of_day") < 17 ), "Afternoon" )
                               .when((col("hour_of_day") >= 17 ) & (col("hour_of_day") < 21 ), "Evening" )
                               .otherwise("Night")
                )
)

StatementMeta(, 40cc943d-a31c-4fa1-b881-8fc335be05b3, 13, Finished, Available, Finished, False)

In [12]:
silver_table_path = 'abfss://WindPowerAnalytics@onelake.dfs.fabric.microsoft.com/LH_Wind_Power_Silver.Lakehouse/Tables/dbo/wind_power'



StatementMeta(, 40cc943d-a31c-4fa1-b881-8fc335be05b3, 14, Finished, Available, Finished, False)

In [13]:
df_transformed.write.format("delta").mode("overwrite").save(silver_table_path)

StatementMeta(, 40cc943d-a31c-4fa1-b881-8fc335be05b3, 15, Finished, Available, Finished, False)

In [14]:
%%sql

select max(date)
from LH_Wind_Power_Silver.dbo.wind_power



StatementMeta(, 40cc943d-a31c-4fa1-b881-8fc335be05b3, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>